# Study 944 — How Much Leverage ⚖️

**Constant leverage on the S&P 500: where is the growth-optimal multiple, and could you
ever have known?**

Hold the index at a fixed multiple `L`, reset daily, financing the borrowed part at
T-bills plus a spread. Growth rises with `L`, then the variance drag `L²σ²/2` swamps it —
so there is a peak, and theory names it: the Kelly / Merton multiple `L* = μ/σ²`. The
practical question is not whether that peak exists. It is whether its *location* is a
number you can estimate today and use tomorrow.

Tape: **SPY** daily **total-return** closes financed at **^IRX** + **50 bps/yr
(a PROXY, swept 0-200)**, 1 bp one-way on the daily reset's turnover,
2003-06-04 → 2026-06-30 (5,799 days). Every Sharpe is excess-of-cash on both sides.

*Real-tape numbers below are the frozen headline (`docs/results.md`, Fingerprint
`d6dfd514b42d`, as-of 2026-06-30). The live cells run the **offline synthetic control** only,
and say so.*


## 1. Why there is a 'best' amount of leverage at all

Double your exposure and you double your average daily return — but you also double your daily swings, and swings *cost* you when you compound. Lose 50% and you need +100% to get level. So the growth of a levered index goes up, flattens, and eventually turns back down. Somewhere there is a peak.

Here is the real curve over 2003-2026, after paying to borrow and after the daily reset's trading costs.

In [1]:
R = dict(lev=[1.0, 1.5, 2.0, 2.5, 2.85, 3.0], tw=[11.68, 22.81, 36.35, 47.17, 49.99, 49.66], cagr=[11.27, 14.56, 16.9, 18.23, 18.53, 18.49], dd=[-55.2, -72.6, -84.2, -91.3, -94.5, -95.5], sharpe=[0.575, 0.566, 0.56, 0.557, 0.555, 0.555], opt=2.85)
print(f"{'leverage':>9}  {'$1 became':>10}  {'growth/yr':>10}  "
      f"{'worst loss':>11}  {'Sharpe':>7}")
for i, L in enumerate(R['lev']):
    mark = '  <-- the peak' if L == R['opt'] else ''
    print(f"{L:9.2f}x  {R['tw'][i]:9.2f}x  {R['cagr'][i]:+9.2f}%  "
          f"{R['dd'][i]:10.1f}%  {R['sharpe'][i]:+7.3f}{mark}")

 leverage   $1 became   growth/yr   worst loss   Sharpe
     1.00x      11.68x     +11.27%       -55.2%   +0.575
     1.50x      22.81x     +14.56%       -72.6%   +0.566
     2.00x      36.35x     +16.90%       -84.2%   +0.560
     2.50x      47.17x     +18.23%       -91.3%   +0.557
     2.85x      49.99x     +18.53%       -94.5%   +0.555  <-- the peak
     3.00x      49.66x     +18.49%       -95.5%   +0.555


## 2. The peak is real — and so is its price

The best multiple over these 23 years was **2.85×**: $1 became **$49.99** instead of $11.68 unlevered. Textbook Kelly, computed on the same tape, says **3.10×** — close, and slightly higher, because the formula ignores the borrowing spread and the fat left tail.

Now read the last two columns of that table again. At the peak your worst loss was **-94.5%** — the account is down to five cents on the dollar — and the **Sharpe ratio actually got slightly worse** (0.575 → 0.555). That is not a coincidence or a quirk of this sample: constant leverage multiplies your excess returns *and* your excess risk by the same number, so it cannot improve risk-adjusted return. All it can do is move you along the same line, and pay the financing on the way.

> 🔬 **For the quants:** gross of the financing spread the excess Sharpe is 0.5752 at 1× and 0.5752 at 3× — identical to four decimals, because `e_L = L·(r − r_f)` exactly. The Sharpe axis is degenerate by construction; only geometric growth can distinguish the multiples.

## 3. So: is 2.85× the answer?

No — and this is the whole study. Resample the 23 years in quarterly blocks and re-solve for the best multiple each time, 1,000 times over. The answers fill the **entire grid**: the 95% interval runs **[1.00, 3.00]**. 44% of resamples say 'as much as you'll let me', 3% say 'none at all'.

Twenty-three years of daily data — about 5,800 observations — and we cannot say whether the right answer is 1× or 3×. The reason is simple arithmetic: the optimum is `average return ÷ variance`, and while variance is easy to measure, the *average return* is the single hardest number to pin down in finance.

## 4. And it moves

Take a five-year window, look back at it with **perfect hindsight**, and ask what leverage *would* have been best. Slide that window through history (217 of them):

- **24%** of windows: the answer is 1.00× — don't lever.
- **54%** of windows: the answer is 3.00× — lever as hard as we allow.

Year-end readings run 1.00 through 2008-2012 (the financial crisis is inside the window), then flip to 3.00 for most of the decade that follows. The underlying Kelly number swings from **-1.5** to **+10.8**. This is the *easy* version of the question — cheating with hindsight — and it still has no stable answer.

In [2]:
years = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
vals = [1.0, 1.0, 1.0, 1.0, 1.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 2.05, 2.85, 2.8, 3.0, 3.0]
print('the best leverage, in hindsight, over the previous five years:')
for y, v in zip(years, vals):
    bar = '#' * int(round(v * 12))
    print(f'  {y}  {v:4.2f}x  {bar}')

the best leverage, in hindsight, over the previous five years:
  2008  1.00x  ############
  2009  1.00x  ############
  2010  1.00x  ############
  2011  1.00x  ############
  2012  1.00x  ############
  2013  3.00x  ####################################
  2014  3.00x  ####################################
  2015  3.00x  ####################################
  2016  3.00x  ####################################
  2017  3.00x  ####################################
  2018  3.00x  ####################################
  2019  3.00x  ####################################
  2020  3.00x  ####################################
  2021  3.00x  ####################################
  2022  2.05x  #########################
  2023  2.85x  ##################################
  2024  2.80x  ##################################
  2025  3.00x  ####################################
  2026  3.00x  ####################################


## 5. The decade hand-off, and the number that really settles it

Split the sample in 2015. The first era's best multiple was **2.20×**; the second era's was **3.00×** (capped — it wanted more). Take one era's answer and use it in the other: the 2015-2026 optimum (3.00×) applied to 2003-2014 earned +10.56%/yr against +8.82%/yr for not levering at all. It stayed ahead — by 1.7 points — while losing -95.5% peak to trough, which is to say the account was gone long before the compounding argument could pay out.

But here is the number that actually settles it, and it is an uncomfortable one. **Change nothing except where the sample starts.** Same code, same end date, same instrument — only the left edge of the window moves:

In [3]:
starts = ['2003-06-04', '2004-01-06', '2005-01-03', '2007-01-03', '2010-01-04']
opt = [2.85, 2.75, 2.65, 2.6, 3.0]
hand = [10.56, 7.07, 5.67, 2.66, 40.49]
unlev = [8.82, 7.81, 7.63, 6.98, 15.38]
edge = [1.73, -0.74, -1.96, -4.32, 25.11]
print('sample starts   best leverage   its answer used in the other era   vs not levering')
for i, s0 in enumerate(starts):
    verdict = 'BEAT it' if edge[i] > 0 else 'LOST to it'
    print(f'{s0:>13}   {opt[i]:12.2f}x   {hand[i]:+29.2f}%   {unlev[i]:+14.2f}%   -> {verdict} by {abs(edge[i]):.2f}%/yr')

sample starts   best leverage   its answer used in the other era   vs not levering
   2003-06-04           2.85x                          +10.56%            +8.82%   -> BEAT it by 1.73%/yr
   2004-01-06           2.75x                           +7.07%            +7.81%   -> LOST to it by 0.74%/yr
   2005-01-03           2.65x                           +5.67%            +7.63%   -> LOST to it by 1.96%/yr
   2007-01-03           2.60x                           +2.66%            +6.98%   -> LOST to it by 4.32%/yr
   2010-01-04           3.00x                          +40.49%           +15.38%   -> BEAT it by 25.11%/yr


Read the last column. Starting the sample in June 2003, one decade's optimal leverage still **beat** not levering in the other decade. Starting it in January 2004 — **seven months later**, on a boundary nobody chose for any economic reason — it **lost**. Start in 2007 and it loses by 4.32 points a year. Start in 2010, after the crash has been deleted from the sample, and the tape cheerfully reports that the best leverage is 3.00× with a Kelly of 4.55 — lever four and a half times, says the data, because nothing bad has happened yet.

'Optimal leverage' is not a property of the market. It is a property of the slice of history you happened to load.

## 6. The tradable version does not rescue it

Estimate `μ/σ²` from the trailing three years, act on it the next day, cap at 3×. It compounds at **19.26%/yr** against **11.36%** for plain buy-and-hold — an advantage of **+6.86%/yr**. But the statistical test on that growth gap gives ***t* = +1.20** with a confidence interval of [-5.6%, +17.5%] — it comfortably includes zero, and the desk's bar is |*t*| ≥ 2.

Worse, the rule spends **64% of its days pinned at the 3× cap**. Loosen the cap and the return rises while the *t*-stat *falls* (+2.78%/yr at cap 1.5 → +6.86% at cap 3.0; *t* +1.74 → +1.20). That is the signature of a volume knob, not a signal.

## 7. Live check — the machinery is honest (offline synthetic)

The cell below is **synthetic, not the real tape**. It builds a make-believe market whose best leverage is *known* to be 2.0, and a second one where the asset earns exactly cash so any leverage is pure waste. Our sweep must find 2.0 in the first and the floor in the second — otherwise the real-tape result would just be a broken tool.

Watch the individual seeds, though. Even in this stationary, made-up world, *forty years* of daily data locates the optimum only to about ±1. That is the study's finding in a laboratory.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from optimal_leverage import data, strategy as st
GRID = np.round(np.arange(0.0, 3.0001, 0.25), 4)
for tag, ss in [('planted best = 2.0', 1.0), ('null (no reward)', 0.0)]:
    opts = [st.synthetic_detect(data.synthetic_daily(signal_strength=ss, seed=944+s)[0],
                                grid=GRID)['opt_lev'] for s in range(8)]
    print(f'{tag:20s} mean {np.mean(opts):.2f}   per-seed {sorted(opts)}')

planted best = 2.0   mean 1.91   per-seed [1.0, 1.0, 1.0, 1.5, 2.25, 2.5, 3.0, 3.0]


null (no reward)     mean 0.44   per-seed [0.0, 0.0, 0.0, 0.0, 0.25, 0.5, 1.0, 1.75]


## Verdict

- **Signal — Weak.** Levering did raise realised growth, in both eras and at every cost assumption, and the curve peaks sensibly (2.85× against a Kelly of 3.10×). But the claim under test is that the peak is a *locatable* number, and it is not: the bootstrap interval for it is the whole grid, the hindsight answer oscillates between 1× and 3×, and the tradable version clears no significance bar at any setting.
- **Tradability — Mirage.** Nothing risk-adjusted is on offer — leverage cannot move the Sharpe ratio, only lower it through financing. The growth is paid for with a -94% drawdown, and the number you would need to know is the one number the data refuses to tell you — it changes sign on seven months of start date.
- **What is honestly true.** The *shape* of the curve is robust. Its *location* is not, and the location is the entire practical question.